#### BPE算法优化
| 可优化点            | 基础版               | 优化后的做法                       |
| -------------- | ------------------- | ---------------------------- |
| 1. 查找受影响的预分词单元 | 每轮遍历全部 `words_list` | 使用倒排索引，只访问包含 `best_pair` 的单元 |
| 2. 更新pair频率    | 每轮重新统计全部pair        | 只更新合并位置左右两侧发生变化的pair         |


In [ ]:
# 回顾一下之前更新Word_list的代码：遍历 words_list 中的每一个预分词单元，看看里面是否包含 (b'o', b'w')，如果包含，就进行 Merge。
# 缺点：对于不包含best_pair的预分词单元，也会进行一次遍历，在数据规模庞大的时候，会成为很明显的性能瓶颈
# for word in words_list:
#     i = 0
#     while i < len(word)-1:
#         if word[i] == best_pair[0] and word[i+1] == best_pair[1]:
#             word[i:i+2] = [new_token]
#             i += 1
#         else:
#             i += 1

# 优化：倒排索引
# 倒排索引：普通的数据存储中，通常是"索引->内容"的映射关系，但是由于我们的需求是通过best_pair找到含有其的预分词单元，所以应该是"内容->索引"
# 具体做法以下用代码解释
words_list = [                              # 预分词列表
    [b'l', b'o', b'w'],                     # 0 -> low
    [b'l', b'o', b'w', b'e', b'r'],         # 1 -> lower
    [b'l', b'o', b'w', b'e', b's', b't'],   # 2 -> lowest
    [b's', b'l', b'o', b'w'],               # 3 -> slow
    [b's', b'l', b'o', b'w', b'e', b'r'],   # 4 -> slower
    [b'h', b'a', b'p', b'p', b'y'],         # 5 -> happy
    [b'd', b'o', b'g'],                     # 6 -> dog
    [b'l', b'o', b'w'],                     # 7 -> low
]
# 对应的倒排索引表格
indices = {
    (b'l', b'o'): {0, 1, 2, 3, 4, 7},
    (b'o', b'w'): {0, 1, 2, 3, 4, 7},
    (b'w', b'e'): {1, 2, 4},
    (b'e', b'r'): {1, 4},
    (b'e', b's'): {2},
    (b's', b't'): {2},
    (b's', b'l'): {3, 4},
    (b'h', b'a'): {5},
    (b'a', b'p'): {5},
    (b'p', b'p'): {5},
    (b'p', b'y'): {5},
    (b'd', b'o'): {6},
    (b'o', b'g'): {6},
}

# 这一部分配合代码如何书写？
tokens = [
    'low', 'lower', 'lowest', 'slow', 'slower', 'happy', 'dog', 'low'
]
# 特殊Token以及预分词在这边就省略掉了，后续完整代码部分会写
# 进行预分词单元数量统计，到时候方便统计每一个字符的数量
from collections import Counter

count_token = Counter()
for token in tokens:
    count_token[token] += 1
# print(count_token)
# 进行每一个预分词单元中的字符进行统计，并且转utf-8

# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
# 这地方一定要用tokens，因为Counter()对象的东西会合并相同项，但是我们目前这一步目的是为了把每一个预分词转换成字节序列
# !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
#=====================================================================================================

# 写法一
word_list = []   # 用列表方便一点
count_list = []
# 也可以用字典，但是比较麻烦
for word in tokens: 
    char = []
    for b in word:
        # count_char_freq[b] += 1 统计全局的，不对，应该先找个东西把分开的字符存起来
        char.append(b.encode('utf-8'))
    word_list.append(char)
    count_list.append(count_token[word])

# =====================================================================================================
# 写法二
word_list_v2 = []
count_list_v2 = []

for word in tokens:
    # print(word, idx) # low 2 lower 1
    # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
    # str → 使用 UTF-8 编码得到 bytes 对象 → 遍历 bytes 时，Python 返回的是每个字节对应的整数（0~255），例如 b'low' 会依次得到 108、111、119 
    # → 使用 bytes([i]) 将整数重新包装成长度为 1 的 bytes 对象，即 b'l'、b'o'、b'w'。其中 bytes() 的参数需要是一个由整数构成的可迭代对象，因此写成 bytes([i])
    # !!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!
    byte_token = [
        bytes([b]) for b in word.encode('utf-8')
        # b.encode('utf-8') for b in word # 不等价写法，因为我们要做的是字节流BPE，而不是字符流
        # "我".encode("utf-8") 得到b'\xe6\x88\x91'然后再拆成[ b'\xe6', b'\x88', b'\x91' ]于是可以统计(b'\xe6', b'\x88')，(b'\x88', b'\x91')经过很多轮 merge，最后又会重新合并成：我。甚至：我们
    ]
    word_list_v2.append(byte_token)
    count_list_v2.append(count_token[word])
# print(word_list_v2)

# 写法三： 字典方式存储，字典没法存'[ ]'列表，所以要存元组，但是存元组必须用tuple转换一下，不然存的是生成器
word_info = {}
for idx, word in enumerate(tokens):
    word_info[idx] = {
        # tuple(bytes([b]) for b in word.encode('utf-8')): freq
        tuple(b.encode('utf-8') for b in word): count_token[word]
    }
# print(word_info) # {0: {(b'l', b'o', b'w'): 2}, 1: {(b'l', b'o', b'w', b'e', b'r'): 1}, 2: {(b'l', b'o', b'w', b'e', b's', b't'): 1}......}

# 转换成字节序列-> 寻找best_pair(就用写法一的两个list了)
total_freq = Counter()
for idx, word in enumerate(word_list):

    for i in range(len(word)-1):
        pair = (word[i],word[i+1])
        total_freq[pair] += count_list[idx]

# 优化开始：
#   优化一：只有包含被 Merge Pair 的那些单词需要更新。（倒排索引）
#   优化二：对于这些单词，只需更新 Merge 位置附近的 Pair。其它单词的 Pair 完全不用重新计算，因此可以直接复用之前的结果。

# 倒排索引：构建字典，pair->拥有目前pair对的预分词单元索引
# 例如： indices = {
#     (b'l', b'o'): {0, 1, 2, 3, 4, 7},
#     (b'w', b'e'): {1, 2, 4},
#     (b'e', b'r'): {1, 4}
# }

from collections import defaultdict, Counter
# defaultdict：与普通的dict不同，当访问一个不存在的键时，会自动创建一个默认值

indices = defaultdict(set)
# print(indices) # defaultdict(<class 'set'>, {})
# 第一次访问的保存过程：indices[(b'l', b'o')] -> indices[(b'l', b'o')] = set() -> 字典由{} -> { (b'l', b'o'): set() }
# 为什么用set，不用list？因为在每一个预分词单元中，相同的pair只需要保存一次，set会自动去重。如果是list的话，indices[(b'a', b'b')] = [5, 5]
# set为集合，用 '{ }'表示，add{0}会输出{0}

total_freq = Counter()

for word_id, word in enumerate(word_list):
    for i in range(len(word) - 1):
        pair = (word[i], word[i + 1])
        indices[pair].add(word_id) # 会遍历每一个预分词单元中的所有pair对组合
        # 还需要统计pair频率了，直接用indices里面的长度不行，因为set去重了
        # "indices 记录的是"哪些不同的预分词单元包含该 Pair"，而 total_freq 记录的是"该 Pair 在整个语料中的加权出现次数"。"
        total_freq[pair] += count_list[word_id]
# print(dict(indices)) # 转换成普通的dict使用就行了
# print(total_freq)

# indices = dict(indices) 多余了
# print(indices.items())

#==================== 一次BPE =================================
# 找到频率最高的pair组合,((b'o', b'w'), 6)，得用索引取出来
best_pair = max(
    total_freq.items(),
    key = lambda item : (item[1],item[0])
)[0]

# new_token = best_pair[0] + best_pair[1]
new_token = b''.join(best_pair)

# 利用倒排索引，只遍历受影响的预分词单元
# affected_words = indices[best_pair] # 利用key，找到value
# for word_id in affected_words:  # 遍历索引，去找每一个预分词单元
#    word = word_list[word_id]
#    i = 0
#    while i < len(word)-1:
#        if word[i] == best_pair[0] and word[i + 1] == best_pair[1]:
#            word[i:i+2] = [new_token] # 先不做，后面跟优化2一起运行就行
#            i += 1
#        else: 
#            i += 1

# 倒排索引已经优化好了。如果要多次更新token，接下来会出现下一个问题
# Merge 完以后，word_list 已经发生了变化，而 pair频率列表 和 倒排索引字典 仍然保存的是 Merge 之前的数据。
# 优化2：更新pair频率列表 和 倒排索引字典。更新原则为：只更新 Merge 位置附近受到影响的 Pair，其它 Pair 可以直接复用。
# 故，优化2是基于优化1的

# affected_words = indices[best_pair] # 利用key，找到value
# 这代码已经没问题了，但是为了防止在循环中进行错误的操作（拷贝一份，避免遍历时修改原集合）
affected_words = list(indices[best_pair])   # 关键改动：加 list()

# print(affected_words) # {0, 1, 2, 3, 4}

for word_id in affected_words:  # 遍历索引，去找每一个预分词单元
    word = word_list[word_id] # 找到对应的预分词单元
    i = 0
    while i < len(word)-1:
        if word[i] == best_pair[0] and word[i + 1] == best_pair[1]:

            # 找 Merge 前受到影响的 Pair（后续用于更新 total_freq 和 indices）
            # 也就是合并前左边和右边的pair
            # =========================
            # Merge 前：删除旧 Pair
            # =========================
            # 左侧 Pair：(x, best_pair[0])
            # 说明合并的pair前有字符，例如要合并的为(b'l', b'o')，即 [b's', b'l', b'o', b'w']中的b's'
            if i > 0: 
                left_pair = ((word[i-1], word[i])) # 确定左边的pair对
                total_freq[left_pair] -= count_list[word_id] # 每一次遇到了就把它的数值剪掉
                if total_freq[left_pair] <= 0:
                    del total_freq[left_pair]
                # ！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！
                # 这里一定要用total_freq做减法，因为只在包含 best_pair 的单词中被破坏，但可能在其他单词中还存在
                # ！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！！
                    # del count_list[word_id] # 这个不删除，因为它保存的是预分词的数量，而不是你这个pair的数量
                # 从 left_pair 的索引中移除当前 word_id
                if left_pair in indices:
                    indices[left_pair].discard(word_id)
                    if len(indices[left_pair]) == 0:
                        del indices[left_pair]


            if i + 2 < len(word): # 说明合并的pair后面有字符，例如要合并的为(b'l', b'o')，即 [b's', b'l', b'o', b'w']中的b'w'
                right_pair = ((word[i+1],word[i+2]))
                total_freq[right_pair] -= count_list[word_id] # 每一次遇到了就把它的数值剪掉
                if total_freq[right_pair] <= 0:
                    del total_freq[right_pair]
                    # del count_list[word_id] # 这个不删除，因为它保存的是预分词的数量，而不是你这个pair的数量
                if right_pair in indices:
                    indices[right_pair].discard(word_id)
                    if len(indices[right_pair]) == 0:
                        del indices[right_pair]
            
            # 这样写是不对的，因为他是一个对象，我们已经在循环遍历affected_words了，相当于在遍历indices[best_pair]
            # 由于已经在对indices[best_pair]循环操作了，所以不能再对indices[best_pair]在循环中做任何操作
            # python不允许这么做，原因就不解释了，记住不行就行
            # indices[best_pair].discard(word_id)
           
            # if len(indices[left_pair]) == 0:
            #     del indices[best_pair]
            
            word[i:i+2] = [new_token] # 更新一下当前频率最高的pair
            
            # =========================
            # Merge 后：增加新 Pair
            # =========================
            if i > 0: # 在左边新增
                new_left_pair = (word[i-1], new_token)
                total_freq[new_left_pair] += count_list[word_id]
                indices[new_left_pair].add(word_id)

            # 由于前面已经合并了，所以现在的len(word)长度会比原来的小1
            if i + 1 < len(word):
                new_right_pair = (new_token, word[i+1])
                total_freq[new_right_pair] += count_list[word_id]
                indices[new_right_pair].add(word_id)                
            

        else: 
            i += 1

if best_pair in indices:
    del indices[best_pair]
if best_pair in total_freq:
    del total_freq[best_pair]


{0, 1, 2, 3, 4, 7}


In [ ]:
# BPE的一次迭代已经结束了，后续训练中需要多次迭代，需要额外进行判断
# 根据当前代码来说的话，需要进行两种情况的判断，但是实际上只有一种

for _ in range(number):
    # 判断是否还有可以合并的pair
    # 方法一
    if not total_freq:
        print(f"第 {merge_step} 步：无相邻 pair，BPE 提前结束")
        break

    # 方法二，完全不好用，撤回...
    if all(len(word) == 1 for word in affected_words): print("无相邻pair，BPE结束") break

    # ======================================================================
    # BPE一次循环
    merge = []

    best_pair = max(
        total_freq.items(),
        key = lambda item : (item[1],item[0])
    )[0]

    # new_token = best_pair[0] + best_pair[1]
    new_token = b''.join(best_pair)
    
    affected_words = list(indices[best_pair])   # 关键改动：加 list()

    for word_id in affected_words:  # 遍历索引，去找每一个预分词单元
        # ...
    
    # 扩展词表，这个是一边找best_pair一边扩展词表
    # 还可以在开头加一个 merge = []，汇总要扩展的token，最后一起扩展。利用这种方式也更方便debug
    # ======================================================================

# 后续词表扩展结束肯定要保存为文件的，这个不要忘记了，要用的时候直接加载就行

